In [9]:
import pandas as pd
import numpy as np
import re
import spacy

In [1]:

# Load cleaned dataset
df = pd.read_csv("../data/processed/asrs_cleaned_3yrs.csv")
print("Loaded:", df.shape)
df.head()


Loaded dataset: (16535, 127)


C:\Users\jenny\AppData\Local\Temp\ipykernel_6328\2300906286.py:6: DtypeWarning: Columns (7,8,15,19,20,41,48,59,63,78,79,83,89,100,110,111,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/asrs_cleaned_3yrs.csv")


,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial,Relative Position.Distance.Nautical Miles,Altitude.AGL.Single Value,Altitude.MSL.Single Value,Latitude / Longitude (UAS),...,Result,Contributing Factors / Situations,Primary Problem,Narrative,Callback,Narrative.1,Callback.1,Synopsis,source_file,TimeOfDayBucket
0,1714400,2020-01,0001-0600,ORD.Airport,IL,NaN,NaN,NaN,15000.0,NaN,...,Air Traffic Control Issued New Clearance; Air ...,Aircraft; Company Policy,Company Policy,To be clear I did not have a lot going on and ...,NaN,NaN,NaN,A TRACON Departure Controller reported two air...,ASRS_DBOnline_012020-062020.csv,Night
1,1714553,2020-01,1801-2400,ZHU.ARTCC,TX,NaN,NaN,NaN,14000.0,NaN,...,Flight Crew Became Reoriented; General Physica...,Human Factors; Environment - Non Weather Related,Environment - Non Weather Related,While descending into PIB out of 14;000 ft. MS...,NaN,NaN,NaN,CRJ Captain reported that a laser shone repeat...,ASRS_DBOnline_012020-062020.csv,Evening
2,1714675,2020-01,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,9000.0,NaN,...,Flight Crew Returned To Departure Airport; Fli...,Aircraft; Weather; Procedure,Weather,I was boarding the aircraft when a my destinat...,NaN,NaN,NaN,Pilot reported weather and a closed destinatio...,ASRS_DBOnline_012020-062020.csv,Evening
3,1714708,2020-01,0601-1200,ZZZ.Airport,US,NaN,NaN,0.0,NaN,NaN,...,General None Reported / Taken,Procedure; Logbook Entry; Human Factors,Ambiguous,I had a ferry flight that went out of service ...,NaN,We departed ZZZ ferrying an un-airworthy aircr...,NaN,Air carrier Dispatcher and flight crew reporte...,ASRS_DBOnline_012020-062020.csv,Morning
4,1714843,2020-01,1801-2400,OSU.Airport,OH,NaN,0.0,0.0,NaN,NaN,...,Flight Crew Diverted; Flight Crew Landed in Em...,Aircraft; Airport,Aircraft,We were flying at 2500 ft. I started losing el...,NaN,NaN,NaN,A Pilot reported a loss of electrical power at...,ASRS_DBOnline_012020-062020.csv,Evening


In [10]:
nlp = spacy.load("en_core_web_sm")

In [11]:
def extract_causal_chain(text):
    if pd.isna(text):
        return np.nan

    steps = []
    keywords = ["before", "during", "after", "then", "subsequently", "eventually", "finally"]

    for kw in keywords:
        if kw in text.lower():
            steps.append(f"Contains temporal marker: {kw}")

    return steps if steps else np.nan

In [12]:
FACTOR_KEYWORDS = {
    "weather": ["turbulence", "wind", "icing", "storm"],
    "communication": ["miscommunication", "did not hear", "unclear", "readback"],
    "fatigue": ["tired", "fatigue", "exhausted"],
    "workload": ["busy", "task saturated", "high workload"],
    "automation": ["autopilot", "automation", "mode confusion"],
}

def extract_contributing_factors(text):
    if pd.isna(text):
        return np.nan

    found = []
    lower = text.lower()

    for factor, words in FACTOR_KEYWORDS.items():
        if any(w in lower for w in words):
            found.append(factor)

    return found if found else np.nan

In [13]:
HFACS = {
    "situational_awareness": ["did not see", "failed to notice", "unaware"],
    "decision_errors": ["should have", "incorrect", "mistake"],
    "skill_based_errors": ["overshoot", "misjudged", "failed"],
    "communication": ["did not hear", "miscommunication", "no response"],
}

def extract_human_factors(text):
    if pd.isna(text):
        return np.nan

    lower = text.lower()
    found = []

    for hf, words in HFACS.items():
        if any(w in lower for w in words):
            found.append(hf)

    return found if found else np.nan

In [14]:
def extract_risk_signals(text):
    if pd.isna(text):
        return np.nan

    signals = []

    if re.search(r"\baltitude\b.*\bdeviation\b", text.lower()):
        signals.append("altitude deviation")

    if re.search(r"loss of separation", text.lower()):
        signals.append("loss of separation")

    if re.search(r"near miss", text.lower()):
        signals.append("near miss")

    if re.search(r"unstable approach", text.lower()):
        signals.append("unstable approach")

    return signals if signals else np.nan

In [15]:
def extract_entities(text):
    if pd.isna(text):
        return np.nan

    doc = nlp(text)
    ents = [f"{ent.text} ({ent.label_})" for ent in doc.ents]
    return ents if ents else np.nan

In [ ]:
extracted_data = []

for idx, row in df.iterrows():
    narrative = row.get("Narrative", "")
    if pd.isna(narrative) or not narrative.strip():
        continue

    try:
        extracted = extract_fields(narrative)
        extracted_data.append({"index": idx, "extracted": extracted})
    except Exception as e:
        extracted_data.append({"index": idx, "extracted": None, "error": str(e)})

In [18]:
# Process in batches of 500 rows
batch_size = 500
results = []

for start in range(0, len(df), batch_size):
    end = start + batch_size
    batch = df.iloc[start:end]

    print(f"Processing rows {start} to {end}...")

    batch_result = batch['Narrative'].apply(lambda text: {
        "causal_chain": extract_causal_chain(text),
        "contributing_factors": extract_contributing_factors(text),
        "human_factors": extract_human_factors(text),
        "risk_signals": extract_risk_signals(text),
        "entities": extract_entities(text)
    })

    results.extend(batch_result.tolist())

Processing rows 0 to 500...
Processing rows 500 to 1000...
Processing rows 1000 to 1500...
Processing rows 1500 to 2000...
Processing rows 2000 to 2500...
Processing rows 2500 to 3000...
Processing rows 3000 to 3500...
Processing rows 3500 to 4000...
Processing rows 4000 to 4500...
Processing rows 4500 to 5000...
Processing rows 5000 to 5500...
Processing rows 5500 to 6000...
Processing rows 6000 to 6500...
Processing rows 6500 to 7000...
Processing rows 7000 to 7500...
Processing rows 7500 to 8000...
Processing rows 8000 to 8500...
Processing rows 8500 to 9000...
Processing rows 9000 to 9500...
Processing rows 9500 to 10000...
Processing rows 10000 to 10500...
Processing rows 10500 to 11000...
Processing rows 11000 to 11500...
Processing rows 11500 to 12000...
Processing rows 12000 to 12500...
Processing rows 12500 to 13000...
Processing rows 13000 to 13500...
Processing rows 13500 to 14000...
Processing rows 14000 to 14500...
Processing rows 14500 to 15000...
Processing rows 15000 to

In [19]:
output_file = "../data/processed/asrs_extracted_offline_3yrs.csv"
df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Final shape:", df.shape)

Saved: ../data/processed/asrs_extracted_offline_3yrs.csv
Final shape: (16535, 131)
